In [1]:
import numpy as np
import pickle
import pandas as pd
import ast
import sys
sys.path.append('../../../../src/')
from utilities import print_exams

In [2]:
def pair_representation(serumHA, virusHA, type, mutate_matrix):
    serumChar = [char for char in serumHA]
    virusChar = [char for char in virusHA]

    if len(serumChar) != len(virusChar):
        return [np.nan for _ in range(len(serumChar))]
    
    return diff_calculation(serumChar, virusChar, type=type, mutate_matrix=mutate_matrix)

def split_data_by_strain(dataframe, identity_cols, frac):
    strains = dataframe[identity_cols].drop_duplicates().reset_index(drop=True)
    test_strains = strains.sample(frac=frac, random_state=42).reset_index(drop=True)
    test_strains_set = set(zip(*test_strains[identity_cols].values.T))

    mask = dataframe[identity_cols].apply(lambda row: tuple(row) in test_strains_set, axis=1)
    test_data = dataframe[mask]
    train_data = dataframe[~mask]

    return train_data, test_data

def diff_calculation(SChar, VChar, type, mutate_matrix):
    difference = []
    for i in range(len(SChar)):
        if type == 'one-hot':
            diff = 0 if SChar[i] == VChar[i] else 1
        if type == 'mut-mat':
            if any(item not in mutate_matrix.columns for item in SChar[i] + VChar[i]):
                diff = 0
            else:
                mut_score = mutate_matrix.loc[VChar[i], VChar[i]]
                ori_score = mutate_matrix.loc[SChar[i], SChar[i]]
                cross_score = mutate_matrix.loc[SChar[i], VChar[i]]
                diff = mut_score + ori_score + cross_score
        difference.append(diff)
    return difference

def run_one_subtype(full_df, subtype, save_path=None, diff_col=None):
    sub_df = full_df[full_df["Type"] == subtype].copy()
   
    group_cols = ["serumName", "virusName", "serumHA", "virusHA", "serumPassCat", "virusPassCat"]
    agg = {c: "first" for c in sub_df.columns if c not in group_cols}
    agg["label"] = "mean"
    sub_df = sub_df.groupby(group_cols, as_index=False).agg(agg)

    meta = sub_df[meta_features].fillna('None').astype('str')

    ohe = OneHotEncoder(handle_unknown='ignore')
    ohe = ohe.fit(meta)
    meta = ohe.transform(meta).toarray()
    seq_diff = np.array(sub_df[diff_col].tolist())[:,16:345]
    data_array = np.hstack((seq_diff, meta))

    nan_mask = pd.isna(data_array).any(axis=1)
    data_array = data_array[~nan_mask]

    X_train, X_valid = train_test_split(data_array, test_size=1/9, random_state=42)
    label_train, label_valid = train_test_split(sub_df['label'].loc[~nan_mask], test_size=1/9, random_state=42)

    print(X_train.shape)
    print(X_valid.shape)

    model_name    = 'AdaBoost'   # the type of model to be used
    model = getattr(Adaboost_utilities, f"model_{model_name}")

    results = model(X_train=np.vstack((X_train, X_valid)), y_train=pd.concat([label_train, label_valid]).tolist())
    
    results['Encoder'] = ohe
    with open(save_path, 'wb') as file:
        pickle.dump(results, file)
    return results


In [3]:
## meta feature for model training
meta_features = ['virusName',   # virus avidity (based on both name and passage)
                'serumName',   # antiserum potency (based on both name and passage)
                'virusPassCat',   # virus passage category
                'serumPassCat']   # serum passage category


In [4]:
def predict_Adaboost_titer(test_df, meta_features, model_path=None):
    # 读取模型和编码器
    with open(model_path, 'rb') as file:
        H1N1_result = pickle.load(file)

    ohe = H1N1_result['Encoder']
    model = H1N1_result['model']

    # 元特征 one-hot
    meta = test_df[meta_features].fillna('None').astype('str')
    meta = ohe.transform(meta).toarray()

    # 提取序列差异特征
    seq_diff = np.array(test_df['seq_diff_ohe'].tolist())[:, 16:345]

    # 拼接数据
    data_array = np.hstack((seq_diff, meta))

    # 去掉含 NaN 的行
    nan_mask = pd.isna(data_array).any(axis=1)
    X_test = data_array[~nan_mask]

    # 标签和预测
    label = test_df['label'].loc[~nan_mask]
    prediction = model.predict(X_test).tolist()

    return label, prediction


## 1. titer test

In [5]:
test_df = pd.read_csv('../../../../data/reverse_test/processed/test_2023SH/test.csv', index_col=False)


test_df['seq_diff_ohe'] = test_df.apply(
        lambda row: pair_representation(row['serumHA'], row['virusHA'], type='one-hot', mutate_matrix=None),
        axis=1
    )

In [6]:
H1N1_label, H1N1_prediction = predict_Adaboost_titer(test_df[test_df['Type'] == 'H1N1'], meta_features, './Adaboost_H1N1_titer.pkl')
H3N2_label, H3N2_prediction = predict_Adaboost_titer(test_df[test_df['Type'] == 'H3N2'], meta_features, './Adaboost_H3N2_titer.pkl')

test_df.loc[test_df['Type'] == 'H1N1', 'pred_with_name'] = H1N1_prediction
test_df.loc[test_df['Type'] == 'H3N2', 'pred_with_name'] = H3N2_prediction
result = print_exams(test_df['pred_with_name'], test_df['label'])

MAE: 0.71659
MSE: 0.95368
pearson correlation: 0.84041
spearman correlation: 0.82402
R2_score: 0.62205


In [7]:
temp_df = test_df.copy()
temp_df.loc[:, 'serumName'] = ''
temp_df.loc[:, 'virusName'] = ''
H1N1_label, H1N1_prediction = predict_Adaboost_titer(temp_df[temp_df['Type'] == 'H1N1'], meta_features, './Adaboost_H1N1_titer.pkl')
H3N2_label, H3N2_prediction = predict_Adaboost_titer(temp_df[temp_df['Type'] == 'H3N2'], meta_features, './Adaboost_H3N2_titer.pkl')

test_df.loc[test_df['Type'] == 'H1N1', 'pred_without_name'] = H1N1_prediction
test_df.loc[test_df['Type'] == 'H3N2', 'pred_without_name'] = H3N2_prediction
result = print_exams(test_df['pred_without_name'], test_df['label'])

MAE: 0.74591
MSE: 1.03290
pearson correlation: 0.82575
spearman correlation: 0.81368
R2_score: 0.60917


In [38]:
test_df.to_csv('../../../Figure/Fig2/Adaboost_titer.csv', index=False)

## 2. strain test

In [23]:
test_df = pd.read_csv('../../../data/data_40/strain/test.csv', index_col=False)
test_df['seq_diff_ohe'] = test_df['seq_diff_ohe'].apply(ast.literal_eval)

In [24]:
H1N1_label, H1N1_prediction = predict_Adaboost_titer(test_df[test_df['Type'] == 'H1N1'], meta_features, model_dir + '/Adaboost_H1N1_strain.pkl')
H3N2_label, H3N2_prediction = predict_Adaboost_titer(test_df[test_df['Type'] == 'H3N2'], meta_features, model_dir + '/Adaboost_H3N2_strain.pkl')

test_df.loc[test_df['Type'] == 'H1N1', 'pred_with_name'] = H1N1_prediction
test_df.loc[test_df['Type'] == 'H3N2', 'pred_with_name'] = H3N2_prediction
result = print_exams(test_df['pred_with_name'], test_df['label'])

MAE: 0.66284
MSE: 0.94046
pearson correlation: 0.87214
spearman correlation: 0.84261
R2_score: 0.75650


In [26]:
temp_df = test_df.copy()
temp_df.loc[:, 'serumName'] = ''
temp_df.loc[:, 'virusName'] = ''
H1N1_label, H1N1_prediction = predict_Adaboost_titer(temp_df[temp_df['Type'] == 'H1N1'], meta_features, model_dir + '/Adaboost_H1N1_strain.pkl')
H3N2_label, H3N2_prediction = predict_Adaboost_titer(temp_df[temp_df['Type'] == 'H3N2'], meta_features, model_dir + '/Adaboost_H3N2_strain.pkl')

test_df.loc[test_df['Type'] == 'H1N1', 'pred_without_name'] = H1N1_prediction
test_df.loc[test_df['Type'] == 'H3N2', 'pred_without_name'] = H3N2_prediction
result = print_exams(test_df['pred_without_name'], test_df['label'])

MAE: 0.86844
MSE: 1.48182
pearson correlation: 0.78572
spearman correlation: 0.73400
R2_score: 0.61634


In [27]:
test_df.to_csv('../../../Figure/Fig2/Adaboost_strain.csv', index=False)

## 3. serum test

In [7]:
test_df = pd.read_csv('../../../data/data_40/serum/test.csv', index_col=False)
test_df['seq_diff_ohe'] = test_df['seq_diff_ohe'].apply(ast.literal_eval)

In [8]:
H1N1_label, H1N1_prediction = predict_Adaboost_titer(test_df[test_df['Type'] == 'H1N1'], meta_features, model_dir + '/Adaboost_H1N1_serum_new.pkl')
H3N2_label, H3N2_prediction = predict_Adaboost_titer(test_df[test_df['Type'] == 'H3N2'], meta_features, model_dir + '/Adaboost_H3N2_serum_new.pkl')

test_df.loc[test_df['Type'] == 'H1N1', 'pred_with_name'] = H1N1_prediction
test_df.loc[test_df['Type'] == 'H3N2', 'pred_with_name'] = H3N2_prediction
result = print_exams(test_df['pred_with_name'], test_df['label'])

MAE: 2.60222
MSE: 9.90406
pearson correlation: -0.07519
spearman correlation: -0.18971
R2_score: -1.52218


In [10]:
temp_df = test_df.copy()
temp_df.loc[:, 'serumName'] = ''
temp_df.loc[:, 'virusName'] = ''
H1N1_label, H1N1_prediction = predict_Adaboost_titer(temp_df[temp_df['Type'] == 'H1N1'], meta_features, model_dir + '/Adaboost_H1N1_serum.pkl')
H3N2_label, H3N2_prediction = predict_Adaboost_titer(temp_df[temp_df['Type'] == 'H3N2'], meta_features, model_dir + '/Adaboost_H3N2_serum.pkl')

test_df.loc[test_df['Type'] == 'H1N1', 'pred_without_name'] = H1N1_prediction
test_df.loc[test_df['Type'] == 'H3N2', 'pred_without_name'] = H3N2_prediction
result = print_exams(test_df['pred_without_name'], test_df['label'])

MAE: 2.59973
MSE: 9.90204
pearson correlation: -0.17028
spearman correlation: -0.33049
R2_score: -1.52167


In [11]:
test_df.to_csv('../../../Figure/Fig2/Adaboost_serum.csv', index=False)

## 4. future test

In [32]:
Crick_41 = pd.read_csv('../../../data/data_40/Crick_41_mapped.csv')
Crick_41['seq_diff_ohe'] = Crick_41['seq_diff_ohe'].apply(ast.literal_eval)

group_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat']
agg_dict = {c: 'first' for c in Crick_41.columns if c not in group_columns}
agg_dict['label'] = 'mean'
test_df = Crick_41.groupby(group_columns).agg(agg_dict).reset_index()

In [10]:
H1N1_label, H1N1_prediction = predict_Adaboost_titer(test_df[test_df['Type'] == 'H1N1'], meta_features, 'Adaboost_H1N1_titer.pkl')
H3N2_label, H3N2_prediction = predict_Adaboost_titer(test_df[test_df['Type'] == 'H3N2'], meta_features, 'Adaboost_H3N2_titer.pkl')

test_df.loc[test_df['Type'] == 'H1N1', 'pred_with_name'] = H1N1_prediction
test_df.loc[test_df['Type'] == 'H3N2', 'pred_with_name'] = H3N2_prediction
result = print_exams(test_df['pred_with_name'], test_df['label'])

MAE: 0.70742
MSE: 0.87159
pearson correlation: 0.67653
spearman correlation: 0.65685
R2_score: 0.38138


In [11]:
temp_df = test_df.copy()
temp_df.loc[:, 'serumName'] = ''
temp_df.loc[:, 'virusName'] = ''
H1N1_label, H1N1_prediction = predict_Adaboost_titer(temp_df[temp_df['Type'] == 'H1N1'], meta_features, 'Adaboost_H1N1_serum.pkl')
H3N2_label, H3N2_prediction = predict_Adaboost_titer(temp_df[temp_df['Type'] == 'H3N2'], meta_features, 'Adaboost_H3N2_serum.pkl')

test_df.loc[test_df['Type'] == 'H1N1', 'pred_without_name'] = H1N1_prediction
test_df.loc[test_df['Type'] == 'H3N2', 'pred_without_name'] = H3N2_prediction
result = print_exams(test_df['pred_without_name'], test_df['label'])

MAE: 0.72625
MSE: 0.96899
pearson correlation: 0.59938
spearman correlation: 0.61475
R2_score: 0.31225


In [12]:
test_df.to_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/Figure/Fig2/Adaboost_41.csv')

## 5. CDC test

In [24]:
CDC_data = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/CDC_all.csv')
CDC_data['seq_diff_ohe'] = CDC_data['seq_diff_ohe'].apply(ast.literal_eval)
CDC_data = CDC_data.loc[CDC_data['virusDate'] <= '2023-08-31',:]

group_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat']
agg_dict = {c: 'first' for c in CDC_data.columns if c not in group_columns}
agg_dict['label'] = 'mean'
test_df = CDC_data.groupby(group_columns).agg(agg_dict).reset_index()

In [25]:
H1N1_label, H1N1_prediction = predict_Adaboost_titer(test_df[test_df['Type'] == 'H1N1'], meta_features, 'Adaboost_H1N1_titer.pkl')
H3N2_label, H3N2_prediction = predict_Adaboost_titer(test_df[test_df['Type'] == 'H3N2'], meta_features, 'Adaboost_H3N2_titer.pkl')

test_df.loc[test_df['Type'] == 'H1N1', 'pred_with_name'] = H1N1_prediction
test_df.loc[test_df['Type'] == 'H3N2', 'pred_with_name'] = H3N2_prediction
result = print_exams(test_df['pred_with_name'], test_df['label'])

MAE: 1.06258
MSE: 2.10216
pearson correlation: 0.75137
spearman correlation: 0.70609
R2_score: 0.50620


In [26]:
temp_df = test_df.copy()
temp_df.loc[:, 'serumName'] = ''
temp_df.loc[:, 'virusName'] = ''
H1N1_label, H1N1_prediction = predict_Adaboost_titer(temp_df[temp_df['Type'] == 'H1N1'], meta_features, 'Adaboost_H1N1_serum.pkl')
H3N2_label, H3N2_prediction = predict_Adaboost_titer(temp_df[temp_df['Type'] == 'H3N2'], meta_features, 'Adaboost_H3N2_serum.pkl')

test_df.loc[test_df['Type'] == 'H1N1', 'pred_without_name'] = H1N1_prediction
test_df.loc[test_df['Type'] == 'H3N2', 'pred_without_name'] = H3N2_prediction
result = print_exams(test_df['pred_without_name'], test_df['label'])

MAE: 1.07081
MSE: 2.16056
pearson correlation: 0.74549
spearman correlation: 0.69442
R2_score: 0.49248


In [27]:
test_df.to_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/Figure/Fig2/Adaboost_CDC.csv')

In [4]:
dd = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/Figure/Fig2/Adaboost_CDC.csv', index_col=False)

In [7]:
print_exams(dd['label'], dd['pred_without_name'])

MAE: 1.07081
MSE: 2.16056
pearson correlation: 0.74549
spearman correlation: 0.69490
R2_score: 0.28890


(1.0708147133702395,
 2.160559495775863,
 PearsonRResult(statistic=0.7454865455169485, pvalue=0.0),
 SignificanceResult(statistic=0.6949038479919861, pvalue=0.0),
 0.2888959866060201)

## 6. CNIC test

In [ ]:
CNIC_data = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/CNIC_all.csv')

In [16]:
CNIC_data = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/CNIC_all.csv')
CNIC_data['seq_diff_ohe'] = CNIC_data['seq_diff_ohe'].apply(ast.literal_eval)
CNIC_data = CNIC_data.loc[CNIC_data['virusDate'] <= '2023-08-31',:]

group_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat']
agg_dict = {c: 'first' for c in CNIC_data.columns if c not in group_columns}
agg_dict['label'] = 'mean'
test_df = CNIC_data.groupby(group_columns).agg(agg_dict).reset_index()

In [17]:
H1N1_label, H1N1_prediction = predict_Adaboost_titer(test_df[test_df['Type'] == 'H1N1'], meta_features, 'Adaboost_H1N1_titer.pkl')
H3N2_label, H3N2_prediction = predict_Adaboost_titer(test_df[test_df['Type'] == 'H3N2'], meta_features, 'Adaboost_H3N2_titer.pkl')

test_df.loc[test_df['Type'] == 'H1N1', 'pred_with_name'] = H1N1_prediction
test_df.loc[test_df['Type'] == 'H3N2', 'pred_with_name'] = H3N2_prediction
result = print_exams(test_df['pred_with_name'], test_df['label'])

MAE: 1.18858
MSE: 2.65191
pearson correlation: 0.70087
spearman correlation: 0.72357
R2_score: 0.46671


In [18]:
temp_df = test_df.copy()
temp_df.loc[:, 'serumName'] = ''
temp_df.loc[:, 'virusName'] = ''
H1N1_label, H1N1_prediction = predict_Adaboost_titer(temp_df[temp_df['Type'] == 'H1N1'], meta_features, 'Adaboost_H1N1_serum.pkl')
H3N2_label, H3N2_prediction = predict_Adaboost_titer(temp_df[temp_df['Type'] == 'H3N2'], meta_features, 'Adaboost_H3N2_serum.pkl')

test_df.loc[test_df['Type'] == 'H1N1', 'pred_without_name'] = H1N1_prediction
test_df.loc[test_df['Type'] == 'H3N2', 'pred_without_name'] = H3N2_prediction
result = print_exams(test_df['pred_without_name'], test_df['label'])

MAE: 1.19454
MSE: 2.64926
pearson correlation: 0.70349
spearman correlation: 0.72118
R2_score: 0.46725


In [20]:
test_df.to_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/Figure/Fig2/Adaboost_CNIC.csv')